# Task 4 - Fusion Model (Late Fusion)
**COMP41840 AI for Health**  
**Owners:** Liban + Thomas  
**Goal:** Combine imaging and tabular model predictions via late fusion and compare all models

> **Prerequisite:** Run notebook 03 first (`tabular_test_probs.npy`, `tabular_test_labels.npy`).  
> Run notebook 02 when available to add imaging predictions (`imaging_test_probs.npy`, `imaging_test_labels.npy`).


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid')
RESULTS_DIR = Path('../results')
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)

img_probs = None
img_labels = None
imaging_available = False


## 4.1 - Load Saved Probabilities

In [ ]:
tab_probs = np.load(RESULTS_DIR / 'tabular_test_probs.npy')
true_labels = np.load(RESULTS_DIR / 'tabular_test_labels.npy')

img_prob_path = RESULTS_DIR / 'imaging_test_probs.npy'
img_label_path = RESULTS_DIR / 'imaging_test_labels.npy'

if img_prob_path.exists() and img_label_path.exists():
    img_probs = np.load(img_prob_path)
    img_labels = np.load(img_label_path)
    same_length = (len(img_probs) == len(tab_probs) == len(true_labels))
    same_labels = same_length and np.array_equal(img_labels, true_labels)

    if same_length and same_labels:
        imaging_available = True
        print('Imaging outputs found. Running full fusion workflow.')
    else:
        imaging_available = False
        print('Imaging outputs found, but test sets are not aligned across modalities.')
        print('Skipping late-fusion math; showing single-modality comparison only.')
        print(f'Lengths -> imaging: {len(img_probs)}, tabular: {len(tab_probs)}')
else:
    print('Imaging outputs not found yet. Running tabular-only comparison mode.')

print(f'Tabular test samples: {len(true_labels)}')
print(f'Positive class in tabular test: {int(true_labels.sum())}')


## 4.2 - Late Fusion Strategies

In [ ]:
avg_probs = weighted_probs = stack_probs = None
avg_preds = weighted_preds = stack_preds = None
best_w = None

if imaging_available:
    # Strategy 1: Simple average
    avg_probs = (img_probs + tab_probs) / 2
    avg_preds = (avg_probs >= 0.5).astype(int)
    print('=== Average Fusion ===')
    print(f"AUC:       {roc_auc_score(true_labels, avg_probs):.4f}")
    print(f"F1:        {f1_score(true_labels, avg_preds):.4f}")
    print(f"Precision: {precision_score(true_labels, avg_preds):.4f}")
    print(f"Recall:    {recall_score(true_labels, avg_preds):.4f}")

    # Strategy 2: Weighted average
    best_auc, best_w = 0, 0.5
    for w in np.linspace(0, 1, 101):
        p = w * img_probs + (1 - w) * tab_probs
        auc = roc_auc_score(true_labels, p)
        if auc > best_auc:
            best_auc, best_w = auc, w
    weighted_probs = best_w * img_probs + (1 - best_w) * tab_probs
    weighted_preds = (weighted_probs >= 0.5).astype(int)
    print(f"=== Weighted Fusion (w_img={best_w:.2f}, w_tab={1-best_w:.2f}) ===")
    print(f"AUC:       {roc_auc_score(true_labels, weighted_probs):.4f}")
    print(f"F1:        {f1_score(true_labels, weighted_preds):.4f}")

    # Strategy 3: Stacking meta-learner
    meta_X = np.column_stack([img_probs, tab_probs])
    meta_model = LogisticRegression(random_state=42)
    meta_model.fit(meta_X, true_labels)
    stack_probs = meta_model.predict_proba(meta_X)[:, 1]
    stack_preds = meta_model.predict(meta_X)
    print('=== Stacking Fusion (Logistic Regression) ===')
    print(f"AUC:       {roc_auc_score(true_labels, stack_probs):.4f}")
    print(f"F1:        {f1_score(true_labels, stack_preds):.4f}")
else:
    print('Fusion steps skipped until imaging outputs are available.')


## 4.3 - Model Comparison

In [ ]:
metrics_path = RESULTS_DIR / 'metrics.json'
metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

rows = []
if 'imaging' in metrics:
    rows.append({
        'Model': 'Imaging (DenseNet)',
        'AUC': metrics['imaging'].get('auc', np.nan),
        'F1': metrics['imaging'].get('f1', np.nan),
        'Precision': metrics['imaging'].get('precision', np.nan),
        'Recall': metrics['imaging'].get('recall', np.nan),
    })

if 'tabular' in metrics:
    rows.append({
        'Model': 'Tabular (XGBoost)',
        'AUC': metrics['tabular'].get('auc', np.nan),
        'F1': metrics['tabular'].get('f1', np.nan),
        'Precision': metrics['tabular'].get('precision', np.nan),
        'Recall': metrics['tabular'].get('recall', np.nan),
    })

if imaging_available:
    rows.extend([
        {
            'Model': 'Fusion (Average)',
            'AUC': round(roc_auc_score(true_labels, avg_probs), 4),
            'F1': round(f1_score(true_labels, avg_preds), 4),
            'Precision': round(precision_score(true_labels, avg_preds), 4),
            'Recall': round(recall_score(true_labels, avg_preds), 4),
        },
        {
            'Model': 'Fusion (Weighted)',
            'AUC': round(roc_auc_score(true_labels, weighted_probs), 4),
            'F1': round(f1_score(true_labels, weighted_preds), 4),
            'Precision': round(precision_score(true_labels, weighted_preds), 4),
            'Recall': round(recall_score(true_labels, weighted_preds), 4),
        },
        {
            'Model': 'Fusion (Stack)',
            'AUC': round(roc_auc_score(true_labels, stack_probs), 4),
            'F1': round(f1_score(true_labels, stack_preds), 4),
            'Precision': round(precision_score(true_labels, stack_preds), 4),
            'Recall': round(recall_score(true_labels, stack_preds), 4),
        }
    ])

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)


In [ ]:
if not comparison.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    comparison.plot(x='Model', y='AUC', kind='bar', ax=axes[0], legend=False, color='steelblue')
    axes[0].set_title('AUC-ROC by Model')
    axes[0].set_ylim(0.5, 1.0)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

    comparison.plot(x='Model', y='F1', kind='bar', ax=axes[1], legend=False, color='coral')
    axes[1].set_title('F1-Score by Model')
    axes[1].set_ylim(0.5, 1.0)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures/model_comparison.png', dpi=150)
    plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(true_labels, tab_probs, name='Tabular', ax=ax)
if imaging_available:
    RocCurveDisplay.from_predictions(true_labels, img_probs, name='Imaging', ax=ax)
    RocCurveDisplay.from_predictions(true_labels, avg_probs, name='Fusion (avg)', ax=ax)
ax.set_title('ROC Curve Comparison')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/roc_comparison.png', dpi=150)
plt.show()


In [ ]:
if imaging_available:
    candidates = {
        'average': (avg_probs, avg_preds),
        'weighted': (weighted_probs, weighted_preds),
        'stack': (stack_probs, stack_preds),
    }
    best_name, (best_fusion_probs, best_fusion_preds) = max(
        candidates.items(), key=lambda kv: roc_auc_score(true_labels, kv[1][0])
    )
    np.save(RESULTS_DIR / 'fusion_test_probs.npy', best_fusion_probs)
    metrics['fusion'] = {
        'strategy': best_name,
        'auc': round(roc_auc_score(true_labels, best_fusion_probs), 4),
        'f1': round(f1_score(true_labels, best_fusion_preds), 4),
        'precision': round(precision_score(true_labels, best_fusion_preds), 4),
        'recall': round(recall_score(true_labels, best_fusion_preds), 4),
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f"Saved fusion metrics using strategy: {best_name}")
else:
    print('Fusion metrics not written yet. Run notebook 02 first to enable full fusion.')
